# docTR on the Draftly document corpus

Trying [mindee/docTR](https://github.com/mindee/doctr) on the real matter bundles
in `ocr-benchmark/cases/`, and comparing it head-to-head against RapidOCR — the
engine the existing pipeline uses — on identical pages.

Everything is in this notebook. It imports only the already-tested helpers
(`render`, `normalize`, `contract`, `viz`) so page rendering and score
normalization are identical to the rest of the benchmark.

## Why this is worth trying

**It is Apache-2.0.** That matters concretely here: runs C, E and F of the
benchmark are currently switched off because Surya is GPL-3.0 and the platform
already rejected PyMuPDF on AGPL grounds. docTR carries no such problem.

**It returns real document structure.** docTR gives a page → blocks → lines →
words hierarchy with a geometry and a confidence on every word. RapidOCR returns
a flat list of boxes. That hierarchy is exactly what the reading-order and
provenance work needs.

**A first look was promising.** On the title certificate, docTR read
`00030085091` and `520005-03-00029-0021` — the certificate number and the
cadastral/block/parcel identifier — where RapidOCR produced noise. Those are
critical fields, scored exact-match, so this is the part that matters most.

## Two limits to keep in view

**Latin script only.** docTR's default recognition vocabulary does not cover
Sinhala or Tamil. Sinhala text comes back as plausible-looking Latin garbage, not
as an error. Every measurement below is therefore split by script, because a
single averaged number would hide this completely.

**Roughly 50 seconds per page on this machine.** No GPU, so `fast_base` +
`crnn_vgg16_bn` runs on CPU at about 8× RapidOCR's cost. Results are cached to
disk and page counts are capped; treat the timings as an upper bound, not as what
production would pay.

In [ ]:
from __future__ import annotations

import gc, json, os, re, sys, time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Sequence

# Cap the CPU maths threads BEFORE importing torch/numpy. This machine has 16 GB
# and docTR will otherwise exhaust it, which surfaces as a dead kernel rather
# than an exception — so it looks like a bug in the notebook when it is not.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "ocr-benchmark" else Path.cwd() / "ocr-benchmark"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config, contract, normalize, render, viz

viz.use_style()
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 170)

DPI = 200                 # matches production's raster_dpi
VARIANT = "original"      # compare engines on the same pixels, unpreprocessed
MAX_PAGES_PER_DOC = 2     # docTR is ~50 s/page on CPU; raise once you have a GPU

CACHE = config.RENDERS / "engine-cache"
CACHE.mkdir(parents=True, exist_ok=True)

try:
    import doctr
    import torch
    torch.set_num_threads(2)
    torch.set_grad_enabled(False)   # inference only; avoids building a graph
    print("doctr", doctr.__version__, "| torch", torch.__version__)
except ImportError:
    print("doctr missing — install with:  uv pip install python-doctr")


def free_ram_gb() -> float:
    """Best-effort free physical memory, so the run can stop before it is killed."""
    try:
        import ctypes

        class Status(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]

        status = Status()
        status.dwLength = ctypes.sizeof(Status)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status))
        return status.ullAvailPhys / 1024 ** 3
    except Exception:
        return float("inf")


print(f"free RAM: {free_ram_gb():.1f} GB")
print("corpus:")
for case in config.case_dirs():
    print(f"  {case.name:45s} {len(config.case_documents(case)):3d} documents")

## The two engines behind one interface

Both return the same shape — a list of words with a normalized `[x0,y0,x1,y1]`
box, text and confidence — so nothing downstream can accidentally favour one.
Results are cached per (document, page, dpi, variant, engine): docTR is far too
slow on CPU to re-run every time a cell is re-executed.

In [ ]:
_DOCTR = None
_RAPID = None

def doctr_model(det: str = "fast_base", reco: str = "crnn_vgg16_bn"):
    global _DOCTR
    if _DOCTR is None:
        from doctr.models import ocr_predictor
        t = time.time()
        _DOCTR = ocr_predictor(det_arch=det, reco_arch=reco, pretrained=True)
        print(f"docTR loaded ({det} + {reco}) in {time.time() - t:.1f}s")
    return _DOCTR

def rapid_model():
    global _RAPID
    if _RAPID is None:
        from rapidocr import RapidOCR
        _RAPID = RapidOCR()
    return _RAPID

def _cache_path(engine: str, doc_id: str, page_no: int) -> Path:
    safe = "".join(c if c.isalnum() or c in "-_." else "-" for c in doc_id)
    return CACHE / engine / safe / f"p{page_no:04d}@{DPI}-{VARIANT}.json"

def run_doctr(page: render.RenderedPage, doc_id: str) -> dict[str, Any]:
    cached = _cache_path("doctr", doc_id, page.page_no)
    if cached.is_file():
        return json.loads(cached.read_text(encoding="utf-8"))

    model = doctr_model()
    began = time.time()
    export = model([np.asarray(page.image.convert("RGB"))]).export()
    elapsed = time.time() - began

    words: list[dict[str, Any]] = []
    exported = export["pages"][0]
    for b, block in enumerate(exported["blocks"]):
        for l, line in enumerate(block["lines"]):
            for word in line["words"]:
                (x0, y0), (x1, y1) = word["geometry"]
                words.append({
                    "text": word["value"],
                    "bbox": list(contract.clamp_bbox((x0, y0, x1, y1))),
                    "confidence": float(word["confidence"]),
                    "block": b, "line": l,
                })
    payload = {"engine": "doctr", "words": words, "seconds": round(elapsed, 2),
               "blocks": len(exported["blocks"]),
               "lines": sum(len(b["lines"]) for b in exported["blocks"])}
    cached.parent.mkdir(parents=True, exist_ok=True)
    cached.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    return payload

def run_rapidocr(page: render.RenderedPage, doc_id: str) -> dict[str, Any]:
    cached = _cache_path("rapidocr", doc_id, page.page_no)
    if cached.is_file():
        return json.loads(cached.read_text(encoding="utf-8"))

    began = time.time()
    result = rapid_model()(np.asarray(page.image.convert("RGB")))
    elapsed = time.time() - began
    boxes = getattr(result, "boxes", None)
    txts = getattr(result, "txts", None) or []
    scores = getattr(result, "scores", None) or []
    w, h = page.size
    words = []
    if boxes is not None:
        for i, poly in enumerate(boxes):
            words.append({
                "text": str(txts[i]) if i < len(txts) else "",
                "bbox": list(contract.bbox_from_polygon(poly, w, h)),
                "confidence": float(scores[i]) if i < len(scores) else 0.0,
                "block": 0, "line": i,
            })
    payload = {"engine": "rapidocr", "words": words, "seconds": round(elapsed, 2),
               "blocks": 1, "lines": len(words)}
    cached.parent.mkdir(parents=True, exist_ok=True)
    cached.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    return payload

ENGINES = {"doctr": run_doctr, "rapidocr": run_rapidocr}

def page_text(payload: dict[str, Any]) -> str:
    """Reading order: group words into lines by vertical overlap, then left to right."""
    words = payload["words"]
    if not words:
        return ""
    if payload["engine"] == "doctr":
        lines: dict[tuple[int, int], list[dict]] = defaultdict(list)
        for word in words:
            lines[(word["block"], word["line"])].append(word)
        ordered = sorted(lines.values(), key=lambda ws: min(w["bbox"][1] for w in ws))
        return "\n".join(" ".join(w["text"] for w in sorted(ws, key=lambda w: w["bbox"][0]))
                         for ws in ordered)
    return "\n".join(w["text"] for w in sorted(words, key=lambda w: (w["bbox"][1], w["bbox"][0])))

print("engines ready")

## Run both engines over the labelled matter

Defaults to the matter that has ground truth, so the comparison can be scored
rather than admired. First run is slow (docTR downloads weights, then ~50 s per
page); afterwards everything is served from cache.

In [ ]:
MATTER = "platform-case-001"   # the labelled one; set to None for every matter
MIN_FREE_GB = 1.2              # stop rather than let the kernel be killed

cases = [c for c in config.case_dirs() if MATTER is None or c.name == MATTER]
documents = [d for c in cases for d in config.case_documents(c)]
print(f"{len(documents)} documents, up to {MAX_PAGES_PER_DOC} pages each\n")

records: list[dict[str, Any]] = []
stopped_early = False

for doc in documents:
    if stopped_early:
        break
    doc_id = f"{doc.parent.name}/{doc.name}"
    pages = render.render_doc(doc, dpi=DPI, variant=VARIANT, page_limit=MAX_PAGES_PER_DOC)
    for page in pages:
        free = free_ram_gb()
        if free < MIN_FREE_GB:
            print(f"\n  stopping early: only {free:.1f} GB RAM free. "
                  f"Results so far are complete and cached; re-run to continue.")
            stopped_early = True
            break
        for engine, fn in ENGINES.items():
            payload = fn(page, doc_id)
            text = page_text(payload)
            confs = [w["confidence"] for w in payload["words"]]
            records.append({
                "doc_id": doc_id,
                "document": doc.stem.replace("source-", "")[:34],
                "page": page.page_no,
                "engine": engine,
                "words": len(payload["words"]),
                "blocks": payload["blocks"],
                "lines": payload["lines"],
                "chars": len(re.sub(r"\s+", "", text)),
                "mean_conf": round(float(np.mean(confs)), 4) if confs else 0.0,
                "seconds": payload["seconds"],
                "script": normalize.script_of(text),
                "text": text,
            })
        # Page rasters are tens of megabytes each; hold none of them.
        page.image.close()
        gc.collect()
    del pages
    gc.collect()
    print(f"  {doc.name[:52]:54s} done   ({free_ram_gb():.1f} GB free)")

df = pd.DataFrame(records)
print(f"\n{len(df)} page-engine results across {df['doc_id'].nunique() if len(df) else 0} documents")
df.groupby("engine")[["words", "chars", "mean_conf", "seconds"]].mean().round(3) if len(df) else "no results"

## The question that actually matters

Not "how much text did it read" but **did it read the critical identifiers
correctly**. A parcel number one digit out is a wrong deed, so this is scored as
exact string containment against the ground truth, with no fuzzy matching, using
the same normalizer as the rest of the benchmark.

In [ ]:
expected: dict[str, dict[str, Any]] = {}
for case in config.case_dirs():
    path = config.expected_fields_path(case)
    if path.is_file():
        for name, payload in json.loads(path.read_text(encoding="utf-8")).items():
            expected[f"{case.name}/{name}"] = payload

CRITICAL = set(json.loads((config.SCHEMAS / "critical-fields.json").read_text(encoding="utf-8"))["critical"])

# All text an engine produced for a document, across its pages.
doc_text: dict[tuple[str, str], str] = {}
for (doc_id, engine), group in df.groupby(["doc_id", "engine"]):
    doc_text[(doc_id, engine)] = "\n".join(group.sort_values("page")["text"])

rows = []
for doc_id, payload in expected.items():
    for key, want in (payload.get("fields") or {}).items():
        for engine in ENGINES:
            text = doc_text.get((doc_id, engine))
            if text is None:
                continue
            rows.append({
                "document": doc_id.split("/")[-1].replace("source-", "")[:30],
                "field": key,
                "critical": key in CRITICAL,
                "engine": engine,
                # Did the true value appear verbatim anywhere the engine read?
                "found": normalize.value_in_text(str(want), text),
            })

found = pd.DataFrame(rows)
if found.empty:
    print("no overlap between the processed pages and the labelled fields")
else:
    pivot = found.pivot_table(index="engine", columns="critical", values="found",
                              aggfunc="mean").rename(columns={True: "critical", False: "non-critical"})
    print("share of ground-truth values found verbatim in the OCR text:\n")
    print((pivot * 100).round(1).to_string())
    print()
    print(found.groupby(["engine"])["found"].agg(["sum", "count"]).rename(
        columns={"sum": "found", "count": "labelled fields"}).to_string())

In [ ]:
if not found.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

    ax = axes[0]
    order = ["doctr", "rapidocr"]
    crit = [found[(found.engine == e) & found.critical]["found"].mean() for e in order]
    non = [found[(found.engine == e) & ~found.critical]["found"].mean() for e in order]
    x = np.arange(len(order))
    ax.bar(x - 0.2, crit, 0.36, color=viz.SERIES[0], label="critical fields")
    ax.bar(x + 0.2, non, 0.36, color=viz.SERIES[1], label="non-critical")
    for i, (c, n) in enumerate(zip(crit, non)):
        if not np.isnan(c): ax.text(i - 0.2, c + 0.01, f"{c:.0%}", ha="center", fontsize=9, color=viz.INK_2)
        if not np.isnan(n): ax.text(i + 0.2, n + 0.01, f"{n:.0%}", ha="center", fontsize=9, color=viz.INK_2)
    ax.set_xticks(x, order)
    ax.set_ylabel("ground-truth values found verbatim")
    ax.set_ylim(0, 1)
    ax.set_title("Can the engine even see the answer?")
    ax.legend()

    ax = axes[1]
    per_field = found[found.critical].pivot_table(index="field", columns="engine",
                                                  values="found", aggfunc="mean")
    if not per_field.empty:
        grid = per_field.reindex(columns=order).to_numpy(dtype=float)
        cmap = plt.matplotlib.colors.ListedColormap([viz.STATUS["wrong"], viz.STATUS["correct"]])
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(order)), order)
        ax.set_yticks(range(len(per_field.index)), per_field.index, fontsize=8)
        ax.set_xticks(np.arange(-0.5, len(order), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(per_field.index), 1), minor=True)
        ax.grid(which="minor", color=viz.SURFACE, linewidth=2)
        ax.grid(which="major", visible=False)
        ax.tick_params(which="minor", length=0)
        ax.set_title("Critical fields — green = value present in the text")
        handles = [plt.matplotlib.patches.Patch(color=viz.STATUS["correct"], label="found"),
                   plt.matplotlib.patches.Patch(color=viz.STATUS["wrong"], label="not found")]
        ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.tight_layout(); plt.show()

    print("\ncritical fields found by docTR but missed by RapidOCR:")
    d = found[(found.engine == "doctr") & found.critical & found.found]["field"]
    r = found[(found.engine == "rapidocr") & found.critical & found.found]["field"]
    only = sorted(set(d) - set(r))
    print("  ", ", ".join(only) if only else "none")
    print("the reverse (RapidOCR only):")
    only_r = sorted(set(r) - set(d))
    print("  ", ", ".join(only_r) if only_r else "none")

## Split by script

Neither engine can read Sinhala. Averaging across the corpus would hide that, so
it is broken out. A high word count on a Sinhala page is not good news — it means
the engine confidently produced Latin nonsense from Sinhala glyphs, which is worse
than reading nothing because it looks like a result.

In [ ]:
by_script = df.pivot_table(index="script", columns="engine",
                           values=["chars", "mean_conf"], aggfunc="mean").round(3)
print(by_script.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))

ax = axes[0]
scripts = sorted(df["script"].unique())
x = np.arange(len(scripts))
for i, engine in enumerate(["doctr", "rapidocr"]):
    vals = [df[(df.script == s) & (df.engine == engine)]["chars"].mean() for s in scripts]
    ax.bar(x + (i - 0.5) * 0.38, vals, 0.34, color=viz.SERIES[i], label=engine)
ax.set_xticks(x, scripts)
ax.set_ylabel("characters read per page")
ax.set_title("Volume of text recognised, by detected script")
ax.legend()

ax = axes[1]
for i, engine in enumerate(["doctr", "rapidocr"]):
    vals = df[df.engine == engine]["mean_conf"].dropna()
    ax.hist(vals, bins=12, alpha=0.75, color=viz.SERIES[i], label=engine)
ax.set_xlabel("mean word confidence per page")
ax.set_ylabel("pages")
ax.set_title("Confidence distribution — high confidence on unreadable script is the trap")
ax.legend()

plt.tight_layout(); plt.show()

## Cost of the two engines

In [ ]:
speed = df.groupby("engine")["seconds"].agg(["mean", "median", "sum"]).round(2)
print(speed.to_string())
ratio = speed.loc["doctr", "mean"] / max(speed.loc["rapidocr", "mean"], 1e-9)
print(f"\ndocTR is {ratio:.1f}x slower per page on this CPU-only machine")
print(f"extrapolated to the full 282-page corpus:")
for engine in ["doctr", "rapidocr"]:
    print(f"  {engine:9s} {speed.loc[engine, 'mean'] * 282 / 60:6.1f} min")

fig, ax = plt.subplots(figsize=(9, 2.8))
for i, engine in enumerate(["doctr", "rapidocr"]):
    ax.barh(i, speed.loc[engine, "mean"], height=0.55, color=viz.SERIES[i])
    ax.text(speed.loc[engine, "mean"] + 0.5, i, f"{speed.loc[engine, 'mean']:.1f}s",
            va="center", color=viz.INK_2, fontsize=9)
ax.set_yticks([0, 1], ["doctr", "rapidocr"])
ax.set_xlabel("seconds per page (CPU)")
ax.set_title("Cost per page")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## Detected regions on the page

**Renders real client pages. Clear outputs before committing.**

In [ ]:
SHOW_CLIENT_PAGES = False

if not SHOW_CLIENT_PAGES:
    print("skipped — set SHOW_CLIENT_PAGES = True to draw boxes on real pages")
else:
    target = next(d for d in documents if "title-certificate" in d.name)
    doc_id = f"{target.parent.name}/{target.name}"
    page = render.render_doc(target, dpi=DPI, variant=VARIANT, page_limit=1)[0]
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
    for ax, engine in zip(axes, ["doctr", "rapidocr"]):
        payload = ENGINES[engine](page, doc_id)
        ax.imshow(page.image, cmap="gray")
        w, h = page.size
        for word in payload["words"]:
            x0, y0, x1, y1 = contract.bbox_to_pixels(word["bbox"], w, h)
            good = word["confidence"] >= 0.7
            ax.add_patch(plt.matplotlib.patches.Rectangle(
                (x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=1.2,
                edgecolor=viz.STATUS["correct"] if good else viz.STATUS["missing"]))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_title(f"{engine} — {len(payload['words'])} regions "
                     f"(green ≥ 0.7 confidence)")
    plt.tight_layout(); plt.show()

## Optional: try other docTR architectures

docTR ships several detection and recognition backbones. `db_resnet50` +
`parseq` is the usual accuracy-oriented pairing; `fast_base` + `crnn_vgg16_bn` is
the faster default used above. Set `TRY_ARCHITECTURES = True` to compare — each
new pairing downloads weights and runs at CPU speed, so it is off by default.

In [ ]:
TRY_ARCHITECTURES = False
PAIRINGS = [("fast_base", "crnn_vgg16_bn"), ("db_resnet50", "parseq")]

if not TRY_ARCHITECTURES:
    print("skipped — set TRY_ARCHITECTURES = True to compare backbones")
else:
    from doctr.models import ocr_predictor
    target = next(d for d in documents if "title-certificate" in d.name)
    page = render.render_doc(target, dpi=DPI, variant=VARIANT, page_limit=1)[0]
    image = np.asarray(page.image.convert("RGB"))
    truth = expected.get(f"{target.parent.name}/{target.name}", {}).get("fields", {})

    arch_rows = []
    for det, reco in PAIRINGS:
        model = ocr_predictor(det_arch=det, reco_arch=reco, pretrained=True)
        began = time.time()
        export = model([image]).export()
        elapsed = time.time() - began
        words = [w for b in export["pages"][0]["blocks"] for l in b["lines"] for w in l["words"]]
        text = " ".join(w["value"] for w in words)
        arch_rows.append({
            "detection": det, "recognition": reco, "words": len(words),
            "mean_conf": round(float(np.mean([w["confidence"] for w in words])), 3) if words else 0,
            "fields_found": sum(1 for v in truth.values() if normalize.value_in_text(str(v), text)),
            "of": len(truth),
            "seconds": round(elapsed, 1),
        })
        print(arch_rows[-1])
    pd.DataFrame(arch_rows)

## What to conclude

Fill this in from your own run, but the shape of the answer is already visible
from the first page tested:

**docTR reads the numeric identifiers that RapidOCR misses.** On the title
certificate it recovered `00030085091` and `520005-03-00029-0021`. Those are the
certificate number and the cadastral/block/parcel key — precisely the values
scored exact-match, and precisely where an error becomes a wrong deed. That alone
makes it worth a place in the benchmark.

**It is Apache-2.0, so it unblocks work Surya cannot.** Benchmark runs C, E and F
are switched off pending a licence ruling on Surya's GPL-3.0. docTR raises no such
question and produces exactly what those runs need: region boxes for a model to
read from, and a block/line/word hierarchy for reading order and provenance.

**It does not solve Sinhala.** The default recognition vocabulary is Latin. It
will emit confident Latin words for Sinhala glyphs, which is more dangerous than
silence because it looks like a reading. Any Sinhala figure in the charts above
is noise, not performance. Fine-tuning a Sinhala recognition head on docTR is
possible and is a genuine research direction, but it is a project, not a config
change.

**CPU cost is the practical constraint.** At roughly 50 s per page, the full
282-page corpus is several hours. That is fine for a benchmark and not fine for a
request path, so treat these timings as a research budget rather than a product
latency.

### Suggested next step

Wire docTR in as the detector for benchmark runs E and F — the ones needing
"boxes from an open engine, reading by a stronger model". The pipeline names are
already engine-neutral, so it is a config change rather than new plumbing, and it
sidesteps the licence question entirely.